# PS1 — VALIDATOR Device / Bus / Serial mapping

Builds a CSV + Excel lookup of **DEVICE_ID ↔ BUS_ID ↔ serial numbers** for VALIDATOR devices.

**Sources**
- `s3://{gold-bucket}/chicago/silver/dim_device/`
- `s3://{gold-bucket}/chicago/silver/hw_config_current/`

**Run order:** Cell 0 → 1 → 2 → 3 → 4 → 5

Outputs land under `ps1_validator_device_map/` in the notebook working directory.

In [ ]:
# CELL 0 — deps
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pandas", "pyarrow", "s3fs", "openpyxl",
])
print("OK")

In [ ]:
# CELL 1 — config (edit these if needed)
import os
from datetime import datetime, timezone

import pandas as pd

GOLD_BUCKET = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
SILVER_PREFIX = "chicago/silver"
SILVER_BASE = f"s3://{GOLD_BUCKET}/{SILVER_PREFIX}"

OUT_DIR = "ps1_validator_device_map"
BASENAME = "validator_device_bus_serial_map"

# True = only mobile bus validators (bus_device_flag=True)
BUS_ONLY = False
# True = keep SCD2 historical dim_device rows (usually leave False)
INCLUDE_HISTORICAL = False

S3_OPTS = {"client_kwargs": {"region_name": os.environ.get("AWS_DEFAULT_REGION", "us-east-1")}}

os.makedirs(OUT_DIR, exist_ok=True)
print(f"Silver base : {SILVER_BASE}")
print(f"Out dir     : {os.path.abspath(OUT_DIR)}")
print(f"BUS_ONLY={BUS_ONLY}  INCLUDE_HISTORICAL={INCLUDE_HISTORICAL}")

In [ ]:
# CELL 2 — helpers + load dim_device / hw_config_current

def _read_parquet(uri: str) -> pd.DataFrame:
    print(f"Reading {uri} ...")
    try:
        return pd.read_parquet(uri, storage_options=S3_OPTS)
    except TypeError:
        return pd.read_parquet(uri)


def _pick(df: pd.DataFrame, cols):
    return [c for c in cols if c in df.columns]


def load_validator_devices(silver_base: str, current_only: bool = True) -> pd.DataFrame:
    dim = _read_parquet(f"{silver_base}/dim_device")
    if current_only and "is_current" in dim.columns:
        dim = dim[dim["is_current"] == True].copy()  # noqa: E712
    elif current_only and "CURRENT_FLAG" in dim.columns:
        dim = dim[dim["CURRENT_FLAG"].astype(str).isin(["1", "True", "true"])].copy()

    if "mars_device_category" not in dim.columns:
        raise RuntimeError("dim_device missing mars_device_category")

    dim = dim[dim["mars_device_category"].astype(str).str.upper().str.strip() == "VALIDATOR"].copy()
    if dim.empty:
        raise RuntimeError("No VALIDATOR rows found in dim_device")

    keep = _pick(dim, [
        "DEVICE_ID", "DEVICE_KEY", "DEVICE_NAME", "DEVICE_SERIAL_NUMBER",
        "BUS_ID", "bus_device_flag", "FACILITY_ID", "FACILITY_NAME",
        "OPERATOR_ID", "OPERATOR_NAME", "mars_device_category",
        "DEVICE_TYPE_NAME", "TRANSIT_MODE_NAME",
    ])
    out = dim[keep].drop_duplicates(subset=["DEVICE_ID"] if "DEVICE_ID" in keep else ["DEVICE_KEY"])
    print(f"VALIDATOR devices (dim_device): {len(out):,}")
    return out


def load_hw_serials(silver_base: str) -> pd.DataFrame:
    try:
        hw = _read_parquet(f"{silver_base}/hw_config_current")
    except Exception as exc:
        print(f"[WARN] hw_config_current not readable: {exc}")
        return pd.DataFrame()

    serial_col = next(
        (c for c in ("COMPONENT_SERIAL_NBR", "SERIAL_NUMBER", "serial_number", "COMPONENT_SERIAL") if c in hw.columns),
        None,
    )
    if serial_col is None:
        print("[WARN] hw_config_current has no serial column — device-level serial only")
        return pd.DataFrame()

    type_col = next(
        (c for c in ("COMPONENT_DESCRIPTION", "COMPONENT_TYPE", "COMPONENT_PART_NBR") if c in hw.columns),
        None,
    )
    key_cols = _pick(hw, ["DEVICE_KEY", "DEVICE_ID"])
    if not key_cols:
        print("[WARN] hw_config_current missing DEVICE_KEY/DEVICE_ID")
        return pd.DataFrame()

    cols = key_cols + [serial_col] + ([type_col] if type_col else [])
    hw = hw[cols].dropna(subset=[serial_col]).copy()
    hw = hw.rename(columns={serial_col: "COMPONENT_SERIAL_NBR"})
    if type_col and type_col != "COMPONENT_TYPE":
        hw = hw.rename(columns={type_col: "COMPONENT_TYPE"})
    elif type_col is None:
        hw["COMPONENT_TYPE"] = None

    dedupe_keys = [c for c in ["DEVICE_KEY", "DEVICE_ID", "COMPONENT_SERIAL_NBR"] if c in hw.columns]
    hw = hw.drop_duplicates(subset=dedupe_keys)
    print(f"hw_config serial rows: {len(hw):,}")
    return hw


devices = load_validator_devices(SILVER_BASE, current_only=not INCLUDE_HISTORICAL)
hw = load_hw_serials(SILVER_BASE)
display(devices.head(3))
print("device columns:", list(devices.columns))

In [ ]:
# CELL 3 — build DEVICE_ID / BUS_ID / serial mapping

def build_mapping(devices: pd.DataFrame, hw: pd.DataFrame, bus_only: bool) -> pd.DataFrame:
    if bus_only and "bus_device_flag" in devices.columns:
        flag = devices["bus_device_flag"]
        devices = devices[(flag == True) | (flag.astype(str).str.lower().isin(["1", "true"]))].copy()  # noqa: E712
        print(f"After bus_device_flag=True filter: {len(devices):,}")

    if hw.empty:
        mapped = devices.copy()
        mapped["COMPONENT_SERIAL_NBR"] = pd.NA
        mapped["COMPONENT_TYPE"] = pd.NA
    else:
        if "DEVICE_KEY" in devices.columns and "DEVICE_KEY" in hw.columns:
            mapped = devices.merge(hw, on="DEVICE_KEY", how="left")
            if "DEVICE_ID_y" in mapped.columns:
                mapped = mapped.drop(columns=["DEVICE_ID_y"]).rename(columns={"DEVICE_ID_x": "DEVICE_ID"})
        elif "DEVICE_ID" in devices.columns and "DEVICE_ID" in hw.columns:
            mapped = devices.merge(hw, on="DEVICE_ID", how="left")
            if "DEVICE_KEY_y" in mapped.columns:
                mapped = mapped.drop(columns=["DEVICE_KEY_y"]).rename(columns={"DEVICE_KEY_x": "DEVICE_KEY"})
        else:
            mapped = devices.copy()
            mapped["COMPONENT_SERIAL_NBR"] = pd.NA
            mapped["COMPONENT_TYPE"] = pd.NA

    ds = mapped["DEVICE_SERIAL_NUMBER"] if "DEVICE_SERIAL_NUMBER" in mapped.columns else pd.Series(pd.NA, index=mapped.index)
    cs = mapped["COMPONENT_SERIAL_NBR"] if "COMPONENT_SERIAL_NBR" in mapped.columns else pd.Series(pd.NA, index=mapped.index)
    mapped["SERIAL_NUMBER"] = cs.where(cs.notna() & (cs.astype(str).str.strip() != ""), ds)

    preferred = [
        "DEVICE_ID", "DEVICE_KEY", "DEVICE_NAME", "BUS_ID", "bus_device_flag",
        "SERIAL_NUMBER", "DEVICE_SERIAL_NUMBER", "COMPONENT_SERIAL_NBR", "COMPONENT_TYPE",
        "FACILITY_ID", "FACILITY_NAME", "OPERATOR_ID", "OPERATOR_NAME",
        "mars_device_category", "DEVICE_TYPE_NAME", "TRANSIT_MODE_NAME",
    ]
    cols = [c for c in preferred if c in mapped.columns] + [c for c in mapped.columns if c not in preferred]
    mapped = mapped[cols].sort_values(
        by=[c for c in ["BUS_ID", "DEVICE_ID", "COMPONENT_SERIAL_NBR"] if c in mapped.columns],
        kind="mergesort",
    ).reset_index(drop=True)
    return mapped


mapped = build_mapping(devices, hw, bus_only=BUS_ONLY)

n_bus = int(mapped["BUS_ID"].notna().sum()) if "BUS_ID" in mapped.columns else 0
n_ser = int(mapped["SERIAL_NUMBER"].notna().sum()) if "SERIAL_NUMBER" in mapped.columns else 0
n_dev = mapped["DEVICE_ID"].nunique() if "DEVICE_ID" in mapped.columns else len(mapped)
print(
    f"Mapping summary: devices={n_dev:,}  rows={len(mapped):,}  "
    f"BUS_ID populated={n_bus:,}  SERIAL_NUMBER populated={n_ser:,}"
)
display(mapped.head(20))

In [ ]:
# CELL 4 — quick checks
print("Null rates (key cols):")
for c in ["DEVICE_ID", "BUS_ID", "SERIAL_NUMBER", "COMPONENT_SERIAL_NBR", "DEVICE_SERIAL_NUMBER", "bus_device_flag"]:
    if c in mapped.columns:
        print(f"  {c:<22} null={mapped[c].isna().mean():.1%}")

if "BUS_ID" in mapped.columns:
    print("\nTop BUS_ID by device count:")
    display(
        mapped.groupby("BUS_ID", dropna=False)["DEVICE_ID"].nunique()
        .sort_values(ascending=False).head(15).rename("n_devices").to_frame()
    )

In [ ]:
# CELL 5 — write CSV + Excel
csv_path = os.path.join(OUT_DIR, f"{BASENAME}.csv")
xlsx_path = os.path.join(OUT_DIR, f"{BASENAME}.xlsx")

mapped.to_csv(csv_path, index=False)
print(f"Wrote CSV  : {os.path.abspath(csv_path)}  ({len(mapped):,} rows)")

# Device-level (1 row / DEVICE_ID) + component expand + bus summary
if "DEVICE_ID" in mapped.columns:
    device_level = (
        mapped.sort_values(
            by=[c for c in ["COMPONENT_SERIAL_NBR"] if c in mapped.columns],
            na_position="last",
            kind="mergesort",
        )
        .drop_duplicates(subset=["DEVICE_ID"], keep="first")
        .copy()
    )
    if "SERIAL_NUMBER" in mapped.columns:
        best_serial = (
            mapped.groupby("DEVICE_ID", as_index=False)["SERIAL_NUMBER"]
            .agg(lambda s: next((x for x in s if pd.notna(x) and str(x).strip()), pd.NA))
        )
        device_level = device_level.drop(columns=["SERIAL_NUMBER"], errors="ignore").merge(
            best_serial, on="DEVICE_ID", how="left"
        )
else:
    device_level = mapped.drop_duplicates()

try:
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        device_level.to_excel(writer, sheet_name="device_level", index=False)
        mapped.to_excel(writer, sheet_name="device_x_component", index=False)
        if "BUS_ID" in mapped.columns and "DEVICE_ID" in mapped.columns:
            bus = (
                mapped.groupby("BUS_ID", dropna=False)
                .agg(
                    n_devices=("DEVICE_ID", "nunique"),
                    n_serials=(
                        "COMPONENT_SERIAL_NBR" if "COMPONENT_SERIAL_NBR" in mapped.columns else "DEVICE_ID",
                        "nunique",
                    ),
                )
                .reset_index()
                .sort_values("n_devices", ascending=False)
            )
            bus.to_excel(writer, sheet_name="bus_summary", index=False)
    print(f"Wrote Excel: {os.path.abspath(xlsx_path)}")
except Exception as exc:
    print(f"[WARN] Excel skipped ({exc}). CSV is ready. Re-run CELL 0 if openpyxl missing.")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
meta_path = os.path.join(OUT_DIR, f"{BASENAME}_meta_{stamp}.txt")
with open(meta_path, "w", encoding="utf-8") as fh:
    fh.write(f"generated_utc={stamp}\n")
    fh.write(f"silver_base={SILVER_BASE}\n")
    fh.write(f"bus_only={BUS_ONLY}\n")
    fh.write(f"devices={n_dev}\n")
    fh.write(f"rows={len(mapped)}\n")
    fh.write(f"bus_id_populated={n_bus}\n")
    fh.write(f"serial_populated={n_ser}\n")
    fh.write(f"csv={os.path.abspath(csv_path)}\n")
    fh.write(f"xlsx={os.path.abspath(xlsx_path)}\n")
print(f"Wrote meta : {os.path.abspath(meta_path)}")
print("Done.")

In [ ]:
# CELL 6 (optional) — download links in Jupyter
from IPython.display import FileLink, display

display(FileLink(csv_path))
if os.path.exists(xlsx_path):
    display(FileLink(xlsx_path))